# Full-run test for CTAB-GAN-Plus

This notebook provides runnable cells to perform a full training run for 'CTAB-GAN-Plus' using the project `TrainTestSplitPipeline`.

Notes:
- The notebook defaults to **running TSTR evaluations** (requires `xgboost`). Set `SKIP_EVALUATIONS = True` to disable.
- Adjust the model config dictionaries below to control epochs / batch sizes for real full runs.
- Each cell is annotated so you can run the cells interactively per dataset/model.
- CTAB-GAN-Plus training can take significant time depending on epochs and dataset size.

In [ ]:
pip install torch scikit-learn tqdm

In [ ]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

# Convenience wrapper to create a pipeline that optionally disables evaluations
def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [ ]:
pip install xgboost

In [ ]:
# User configuration: choose dataset(s) and run options
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']
SKIP_EVALUATIONS = False  # Set to True to skip TSTR evaluation and run faster

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

MODEL_MAP = {
    'ctabgan': ('katabatic.models.ctabgan.models', 'CTABGANModel'),
}

# CTAB-GAN-Plus Configuration per dataset
# Adult dataset configuration
CTABGAN_CONFIG_ADULT = {
    'epochs': 150,
    'batch_size': 500,
    'random_dim': 100,
    'num_channels': 64,
    'class_dim': (256, 256, 256, 256),
    'l2scale': 1e-5,
    'n_clusters': 10,
    'eps': 0.005,
    'categorical': [1, 3, 5, 6, 7, 8, 9, 13],  # workclass, education, marital-status, occupation, relationship, race, sex, native-country
    'integer': [0, 4, 10, 11, 12],  # age, fnlwgt, capital-gain, capital-loss, hours-per-week
    'problem_type': {'Classification': 'class'},
    'seed': 42,
}

# Car dataset configuration
CTABGAN_CONFIG_CAR = {
    'epochs': 150,
    'batch_size': 500,
    'random_dim': 100,
    'num_channels': 64,
    'class_dim': (256, 256, 256, 256),
    'l2scale': 1e-5,
    'categorical': [0, 1, 2, 3, 4, 5],  # All features are categorical
    'problem_type': {'Classification': '6'},
    'seed': 42,
}

# Magic dataset configuration
CTABGAN_CONFIG_MAGIC = {
    'epochs': 150,
    'batch_size': 500,
    'random_dim': 100,
    'num_channels': 64,
    'class_dim': (256, 256, 256, 256),
    'l2scale': 1e-5,
    'categorical': [],  # All features are continuous
    'problem_type': {'Classification': 'class'},
    'seed': 42,
}

# Nursery dataset configuration
CTABGAN_CONFIG_NURSERY = {
    'epochs': 150,
    'batch_size': 500,
    'random_dim': 100,
    'num_channels': 64,
    'class_dim': (256, 256, 256, 256),
    'l2scale': 1e-5,
    'categorical': [0, 1, 2, 3, 4, 5, 6, 7],  # All features are categorical
    'problem_type': {'Classification': '8'},
    'seed': 42,
}

# Shuttle dataset configuration
CTABGAN_CONFIG_SHUTTLE = {
    'epochs': 150,
    'batch_size': 500,
    'random_dim': 100,
    'num_channels': 64,
    'class_dim': (256, 256, 256, 256),
    'l2scale': 1e-5,
    'categorical': [],  # All features are continuous
    'problem_type': {'Classification': 'class'},
    'seed': 42,
}

# Map datasets to their configs
DATASET_CONFIGS = {
    'adult': CTABGAN_CONFIG_ADULT,
    'car': CTABGAN_CONFIG_CAR,
    'magic': CTABGAN_CONFIG_MAGIC,
    'nursery': CTABGAN_CONFIG_NURSERY,
    'shuttle': CTABGAN_CONFIG_SHUTTLE,
}

## Preprocess datasets (run once)
Run this cell to discretize the raw CSVs into `discretized_data/{dataset}.csv`. 

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(file_path=f'raw_data/{dataset}.csv', output_path=f'discretized_data/{dataset}.csv', bins=10, strategy='uniform')
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        import traceback; traceback.print_exc()

## Run full CTAB-GAN-Plus
This cell runs CTAB-GAN-Plus for each selected dataset using `DATASET_CONFIGS` above. Be patient — full training can take time depending on `epochs` and dataset size.

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'CTAB-GAN-Plus -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'ctabgan')
    ensure(synth_dir)
    try:
        # Get configuration for this dataset
        config = DATASET_CONFIGS[dataset]
        
        mod_path, cls_name = MODEL_MAP['ctabgan']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda cfg=config: ModelClass(**cfg)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('CTAB-GAN-Plus finished for', dataset)
    except Exception as e:
        print('CTAB-GAN-Plus failed for', dataset, e)
        import traceback; traceback.print_exc()